# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:

1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: _my intuition was rough, the model found the real signal, and I can read exactly what it found._


## 0. Setup (Colab or local)

**Method, step by step:** the first cell finds the project's top folder (cloning it on Colab, where the notebook starts on an empty machine), loads the 30,000-page starter CSV, and defines the label: a page is _declining_ when its trend is down. The second cell loads the tools and defines `show()` and `show_table()`, so every result is a titled table in the same layout, plus the chart theme used by the one chart below.


In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

In [ ]:
import subprocess
import tempfile
import urllib.request

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib import font_manager

pd.set_option("display.width", 160)
NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))

# Emphasis theme: cividis navy for the story, grays for what should recede.
INK, MUTED, FAINT, GRID, SURFACE = "#111111", "#5f5e5a", "#9a9893", "#eeede9", "#ffffff"
NAVY, GRAY, PALE = mpl.colormaps["cividis"](0.05), "#9a9893", "#d4d2cc"


def load_inter() -> str:
    """Register the Inter font for this session and return the font family to draw with.

    The 400 and 600 weights are cached in the system temp folder, never in the project.
    If the download is unavailable, the chart falls back to matplotlib's default font,
    so the notebook still runs offline.
    """
    cache = os.path.join(tempfile.gettempdir(), "inter-font")
    os.makedirs(cache, exist_ok=True)
    try:
        for weight in (400, 600):
            path = os.path.join(cache, f"inter-{weight}.ttf")
            if not os.path.exists(path):
                url = f"https://cdn.jsdelivr.net/fontsource/fonts/inter@latest/latin-{weight}-normal.ttf"
                urllib.request.urlretrieve(url, path)
            font_manager.fontManager.addfont(path)
    except OSError:
        return "DejaVu Sans"
    return "Inter"


plt.rcParams.update({"font.family": load_inter()})

## 1. A rule you write by hand: `stale x visible`

Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

**Method:** this cell gives every page a score from the hand rule: it is zero unless the page is both stale and visible, and then it equals the page's impressions. It lists the ten highest-scoring pages and counts how many pages score above zero at all.


In [ ]:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
show_table("The ten pages the hand rule scores highest", top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]])
show("How much of the data the hand rule can rank", {
    "pages with a score above zero": f"{(df['hand_rule_score'] > 0).sum():,}",
    "pages scored exactly zero (tied)": f"{(df['hand_rule_score'] == 0).sum():,}",
    "of the scored pages, how many are declining": f"{df.loc[df['hand_rule_score'] > 0, 'is_declining_label'].sum()} of {(df['hand_rule_score'] > 0).sum()}",
})

We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.


**Method:** this cell defines `precision_at_k`. One detail matters: when many pages share the same score, the order among them is arbitrary, and a naive version would quietly let row order decide the answer. This version fills the slots left at the cutoff with the **average outcome of the tied pages**, so the number does not depend on how the table happens to be sorted. It then scores the hand rule at K = 20 and K = 50 and sets the base rate beside it.


In [ ]:
def precision_at_k(scores, labels, k: int) -> float:
    """Share of the top `k` pages that are declining, with tied scores resolved by their average.

    Pages with equal scores have no real order. The slots left at the cutoff are filled with the
    average label of the tied group, so the result does not depend on row order.
    """
    scores, labels = np.asarray(scores, dtype=float), np.asarray(labels, dtype=float)
    cutoff = np.sort(scores)[::-1][k - 1]
    above, tied = scores > cutoff, scores == cutoff
    return float((labels[above].sum() + (k - above.sum()) * labels[tied].mean()) / k)


def tied_slots(scores, k: int) -> int:
    """How many of the top `k` slots are filled by pages whose score is tied at the cutoff."""
    scores = np.asarray(scores, dtype=float)
    return int(k - (scores > np.sort(scores)[::-1][k - 1]).sum())


y = df["is_declining_label"].to_numpy()
rule_scores = df["hand_rule_score"].to_numpy()
show_table("The hand rule's precision on all 30,000 pages", pd.DataFrame({
    "Precision": {
        "Hand rule at K = 20": f"{precision_at_k(rule_scores, y, 20):.3f}",
        "Hand rule at K = 50": f"{precision_at_k(rule_scores, y, 50):.3f}",
        "Picking pages at random (base rate)": f"{y.mean():.3f}",
    },
    "Slots filled by tied pages": {
        "Hand rule at K = 20": f"{tied_slots(rule_scores, 20)} of 20",
        "Hand rule at K = 50": f"{tied_slots(rule_scores, 50)} of 50",
        "Picking pages at random (base rate)": "all",
    },
}).rename_axis("Scorer"))

## 2. Let a model learn the rule — then read it

A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.


**Method:** this cell gives the tree six signals that are knowable before any decision (page age, days since the last update, impressions, average position, CTR and word count), lets it ask at most two questions, and shows the result twice: as a diagram, where each box is a yes/no question with the pages that reach it and the share of them that are declining (follow the left branch for yes and the right for no; navy leaves say declining and pale leaves say not declining), and as a table with one row per leaf giving the questions that reach it, how many pages end there, what share of them are declining, and what the tree says. The `class_weight` setting makes the tree treat the declining and non-declining pages as equally important.


In [ ]:
from sklearn.tree import DecisionTreeClassifier


def draw_tree(model, data, labels, names: list[str], title: str | None = None) -> None:
    """Draw a fitted decision tree as boxes: each question, the pages that reach it, and how many are declining.

    Internal boxes hold the yes/no question, with yes on the left branch. Leaves are navy when the
    tree says declining and pale when it says not declining.
    """
    nodes = model.tree_
    reach = model.decision_path(data).tocsc()
    members = {node: reach[:, node].nonzero()[0] for node in range(nodes.node_count)}
    position, level, leaves = {}, {}, []
    new_line = chr(10)

    def place(node: int, depth: int) -> float:
        level[node] = depth
        left, right = nodes.children_left[node], nodes.children_right[node]
        if left == -1:
            leaves.append(node)
            position[node] = float(len(leaves) - 1)
        else:
            position[node] = (place(left, depth + 1) + place(right, depth + 1)) / 2
        return position[node]

    place(0, 0)
    says = {leaf: model.classes_[nodes.value[leaf][0].argmax()] == 1 for leaf in leaves}
    if title is None:
        declining_pages = sum(len(members[leaf]) for leaf in leaves if says[leaf])
        title = f"{declining_pages:,} of {len(data):,} pages land in a leaf that says declining"

    rows = max(level.values()) + 1
    fig, ax = plt.subplots(figsize=(max(9.5, 3.0 * len(leaves)), 1.9 + 1.75 * rows), dpi=120, facecolor=SURFACE)
    fig.subplots_adjust(left=0.02, right=0.98, top=0.90, bottom=0.03)
    ax.set_facecolor(SURFACE)
    ax.axis("off")
    ax.set_xlim(-0.7, len(leaves) - 0.3)
    ax.set_ylim(-rows + 0.45, 0.4)

    for node in range(nodes.node_count):
        for child, answer in ((nodes.children_left[node], "yes"), (nodes.children_right[node], "no")):
            if child == -1:
                continue
            ax.plot([position[node], position[child]], [-level[node], -level[child]], color=FAINT, linewidth=1.8,
                    zorder=1, solid_capstyle="round")
            ax.text((position[node] + position[child]) / 2, -(level[node] + level[child]) / 2, answer, ha="center",
                    va="center", color=MUTED, fontsize=10.5, zorder=2,
                    bbox={"facecolor": SURFACE, "edgecolor": "none", "pad": 1.5})

    for node in range(nodes.node_count):
        pages, share = len(members[node]), labels[members[node]].mean()
        if nodes.children_left[node] == -1:
            declining = says[node]
            label = new_line.join([("Declining" if declining else "Not declining"), f"{pages:,} pages",
                                   f"{share:.1%} are declining"])
            fill, edge, ink = (NAVY, NAVY, "white") if declining else ("#f3f2ee", PALE, INK)
            weight = "semibold"
        else:
            name, cut = names[nodes.feature[node]], nodes.threshold[node]
            label = new_line.join([f"{name} <= {cut:,.2f}?", f"{pages:,} pages, {share:.1%} declining"])
            fill, edge, ink, weight = SURFACE, GRAY, INK, "regular"
        ax.text(position[node], -level[node], label, ha="center", va="center", color=ink, fontsize=11.5,
                fontweight=weight, linespacing=1.5, zorder=3,
                bbox={"boxstyle": "round,pad=0.6,rounding_size=0.4", "facecolor": fill, "edgecolor": edge, "linewidth": 1.6})
    fig.text(0.5, 0.955, title, ha="center", color=INK, fontsize=17, fontweight="semibold")
    plt.show()


def leaf_table(model, data, labels, names: list[str]) -> pd.DataFrame:
    """List every leaf of a fitted tree: the questions that reach it, its pages, and what it predicts."""
    nodes, leaf_of = model.tree_, model.apply(data)
    rows = []

    def walk(node: int, path: list[str]) -> None:
        if nodes.children_left[node] == -1:
            mask = leaf_of == node
            predicted = model.classes_[nodes.value[node][0].argmax()]
            rows.append({
                "Questions answered to reach it": " and ".join(path) or "none (every page)",
                "Pages": f"{mask.sum():,}",
                "Share declining": f"{labels[mask].mean():.1%}" if mask.any() else "n/a",
                "Tree says": "declining" if predicted == 1 else "not declining",
            })
            return
        name, cut = names[nodes.feature[node]], nodes.threshold[node]
        walk(nodes.children_left[node], path + [f"{name} <= {cut:,.2f}"])
        walk(nodes.children_right[node], path + [f"{name} > {cut:,.2f}"])

    walk(0, [])
    table = pd.DataFrame(rows)
    table.index = pd.RangeIndex(1, len(table) + 1, name="Leaf")
    return table


features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

draw_tree(tree, X, y, features)
show_table("Where the tree sends every page", leaf_table(tree, X, y, features))

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.


**Method:** this cell scores the tree's probability for every page and compares it with the hand rule at K = 20 and K = 50, in the same table and with the same tie handling. It also counts how many distinct scores each scorer can give, because a depth-2 tree has at most four leaves.


In [ ]:
tree_scores = tree.predict_proba(X)[:, 1]
show_table("Hand rule against the depth-2 tree, scored on the pages it learned from", pd.DataFrame({
    "Precision@20": {"Hand rule": f"{precision_at_k(rule_scores, y, 20):.3f}", "Depth-2 tree": f"{precision_at_k(tree_scores, y, 20):.3f}"},
    "Precision@50": {"Hand rule": f"{precision_at_k(rule_scores, y, 50):.3f}", "Depth-2 tree": f"{precision_at_k(tree_scores, y, 50):.3f}"},
    "Slots at K = 50 filled by ties": {"Hand rule": f"{tied_slots(rule_scores, 50)} of 50", "Depth-2 tree": f"{tied_slots(tree_scores, 50)} of 50"},
    "Distinct scores given": {"Hand rule": f"{len(np.unique(rule_scores)):,}", "Depth-2 tree": f"{len(np.unique(tree_scores))}"},
}).rename_axis("Scorer"))

Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.


## 3. Why you can't feed the outcome back in

Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:


**Method:** this cell adds `trend_pct`, the exact percentage change the label is built from, to the six features and fits the same depth-2 tree. It reports the leaky tree's Precision@50 next to the honest one, names the first question the leaky tree asks, and shows the tree and its leaves in the same two forms as before, so you can see where it sends the pages.


In [ ]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
leaky_scores = leaky.predict_proba(X_leaky)[:, 1]
leaky_names = features + ["trend_pct"]
first = leaky.tree_
show("The leaky tree against the honest one", {
    "honest tree Precision@50": f"{precision_at_k(tree_scores, y, 50):.3f}",
    "leaky tree Precision@50": f"{precision_at_k(leaky_scores, y, 50):.3f}",
    "first question the leaky tree asks": f"{leaky_names[first.feature[0]]} <= {first.threshold[0]:.2f}",
})
draw_tree(leaky, X_leaky, y, leaky_names,
          title=f"The leaky tree's first question: {leaky_names[first.feature[0]]} <= {first.threshold[0]:.2f}")
show_table("Where the leaky tree sends every page", leaf_table(leaky, X_leaky, y, leaky_names))

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable _before_ the outcome.

> Rule of thumb: if a feature would only be known _because someone already made the decision you're predicting_, it leaks. Leave it out.


## 4. 🔧 Your turn

- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored _in-sample_ here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.


**My choice: the third bullet, the held-out client split.** It is the one the notebook calls important, and it decides whether the comparison above means anything. The depth and feature experiments change the tree; this one changes how the tree is judged.


### My experiment: does the gap hold on clients the tree has never seen?

**Method, step by step:**

1. **Splits the pages by client** 20 different times, holding 30% of the clients out as a test group each time. A client's pages are never on both sides of a split.
2. **Fits the depth-2 tree** on the training clients only, then scores the held-out clients with it. The hand rule needs no training, so it is simply scored on the same held-out pages.
3. **Measures Precision@20 and Precision@50** for both scorers on every split, with the same tie handling as before, and records the base rate of each test group so the scores can be read against it.
4. **Summarises** the mean, lowest and highest score over the 20 splits and counts how often the tree was ahead.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

N_SPLITS, TEST_SIZE = 20, 0.3
splitter = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=42)

rows = []
for train_idx, test_idx in splitter.split(X, y, groups=df["client_id"]):
    held = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X.iloc[train_idx], y[train_idx])
    held_scores = held.predict_proba(X.iloc[test_idx])[:, 1]
    labels, rule = y[test_idx], rule_scores[test_idx]
    rows.append({
        "pages": len(test_idx), "scored_by_rule": int((rule > 0).sum()), "base_rate": labels.mean(),
        "rule_20": precision_at_k(rule, labels, 20), "tree_20": precision_at_k(held_scores, labels, 20),
        "rule_50": precision_at_k(rule, labels, 50), "tree_50": precision_at_k(held_scores, labels, 50),
    })
splits = pd.DataFrame(rows)

measures = ["rule_20", "tree_20", "rule_50", "tree_50", "base_rate"]
summary = pd.DataFrame({
    "Mean": splits[measures].mean(),
    "Lowest": splits[measures].min(),
    "Highest": splits[measures].max(),
}).map("{:.3f}".format)
summary.index = ["Hand rule Precision@20", "Tree Precision@20", "Hand rule Precision@50", "Tree Precision@50", "Base rate of the test clients"]
summary.index.name = "Measure"
show_table(f"Held-out client splits ({N_SPLITS} splits, {TEST_SIZE:.0%} of clients held out each time)", summary)
show("How the 20 splits compare", {
    "splits where the tree's Precision@50 is higher": f"{(splits['tree_50'] > splits['rule_50']).sum()} of {N_SPLITS}",
    "splits where the rule's Precision@50 is higher": f"{(splits['rule_50'] > splits['tree_50']).sum()} of {N_SPLITS}",
    "typical test group": f"{splits['pages'].median():,.0f} pages",
    "pages the hand rule can score in a test group (median)": f"{splits['scored_by_rule'].median():,.0f}",
})

In [ ]:
# Chart: each scorer's Precision@50 in-sample against its mean on held-out clients.
chart = {
    "Hand rule": {"color": "tab:orange", "before": precision_at_k(rule_scores, y, 50), "after": splits["rule_50"].mean()},
    "Depth-2 tree": {"color": NAVY, "before": precision_at_k(tree_scores, y, 50), "after": splits["tree_50"].mean()},
}
base = float(y.mean())

fig, ax = plt.subplots(figsize=(10, 4.2), dpi=120, facecolor=SURFACE)
fig.subplots_adjust(left=0.20, right=0.95, top=0.76, bottom=0.12)
ax.set_facecolor(SURFACE)
for row, (name, item) in enumerate(reversed(list(chart.items()))):
    start, end = item["before"], item["after"]
    ax.annotate("", xy=(end, row), xytext=(start, row), zorder=2,
                arrowprops={"arrowstyle": "-|>", "color": item["color"], "lw": 2.6, "mutation_scale": 16, "shrinkA": 8, "shrinkB": 8})
    ax.scatter(start, row, s=170, facecolor=SURFACE, edgecolor=item["color"], linewidth=2.4, zorder=3)
    ax.scatter(end, row, s=170, facecolor=item["color"], edgecolor=SURFACE, linewidth=2, zorder=3)
    for x, other in ((start, end), (end, start)):
        side = 1 if x >= other else -1
        ax.text(x + side * 0.02, row, f"{x:.2f}", ha="left" if side > 0 else "right", va="center", color=INK, fontsize=12.5,
                fontweight="semibold", zorder=4, bbox={"facecolor": SURFACE, "edgecolor": "none", "pad": 2})
    ax.text(0.385, row, name, ha="right", va="center", color=INK, fontsize=12.5, fontweight="semibold")
ax.axvline(base, color=FAINT, linestyle=(0, (4, 4)), linewidth=1.4, zorder=1)
ax.text(base + 0.008, 1.55, f"random pick {base:.2f}", ha="left", va="bottom", color=MUTED, fontsize=10.5, zorder=4)
ax.set_xlim(0.40, 0.80)
ax.set_ylim(-0.6, 1.9)
ax.set_xticks([0.4, 0.5, 0.6, 0.7, 0.8], ["0.4", "0.5", "0.6", "0.7", "0.8"])
ax.set_yticks([])
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_axisbelow(True)
ax.tick_params(axis="x", length=0, colors=FAINT, labelsize=10, pad=8)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.text(0.5, 0.88, f"On clients it never saw, the tree scores {chart['Depth-2 tree']['after']:.2f} and the rule {chart['Hand rule']['after']:.2f}",
         ha="center", color=INK, fontsize=17, fontweight="semibold")
plt.show()

## Limitations

- **The label is a trend bucket, not a decision.** A page is "declining" because `trend_direction` says `down`; nothing here shows that refreshing such a page would bring traffic back.
- **Twenty splits are not twenty independent tests.** They are drawn from the same 32 clients, so the spread between them understates how much a genuinely new set of clients could differ.
- **A depth-2 tree is deliberately small.** It can give only four different scores, so any top-50 ranking built on it is mostly a block of ties; a bigger model can separate pages the tree cannot.
- **Observed, not causal.** The tree reports which signals went with declining pages in this sample, not what makes pages decline.


### Save your work

**Colab:** _File → Save a copy in GitHub_ (your submission) and _File → Save a copy in Drive_.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.
